# Export the tone model for the browser (Colab)

**Why:** free Hugging Face accounts can only host *static* pages, so the demo runs the model in the
visitor's browser with transformers.js. That needs the model in ONNX format, made smaller by
quantization (8-bit weights, about 110 MB instead of 440 MB).

**Steps**
1. A normal CPU runtime is enough; no GPU needed.
2. Run the cells from top to bottom (Shift+Enter).
3. The checks in cells 5 and 6 must pass before you upload anything.
4. The last cell makes the model public. Read the note above it first.

This notebook never reads your gold labels.

In [ ]:
!pip -q install onnx onnxruntime

In [ ]:
from huggingface_hub import notebook_login

notebook_login()   # same as before: follow the link, or paste a token with WRITE access

In [ ]:
import numpy as np
import torch
from huggingface_hub import HfApi
from transformers import AutoModelForSequenceClassification, AutoTokenizer

REPO = "Kobichris/cb-policy-tone"
DATA_REPO = "Kobichris/cb-policy-tone-data"
api = HfApi()

tokenizer = AutoTokenizer.from_pretrained(REPO)
model = AutoModelForSequenceClassification.from_pretrained(REPO, attn_implementation="eager").eval()
print("labels:", model.config.id2label)
files = api.list_repo_files(REPO)
print("tokenizer.json in the repo:", "tokenizer.json" in files)   # transformers.js needs it
assert tokenizer.is_fast, "expected a fast tokenizer"

**Export to ONNX** and check it gives the same numbers as the PyTorch model.

In [ ]:
import os
import onnxruntime as ort

os.makedirs("export/onnx", exist_ok=True)
FP32 = "export/model_fp32.onnx"
NAMES = ["input_ids", "attention_mask", "token_type_ids"]


class Logits(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, input_ids, attention_mask, token_type_ids):
        return self.m(input_ids=input_ids, attention_mask=attention_mask,
                      token_type_ids=token_type_ids).logits


EXPORT = [
    "Headline inflation rose sharply on the back of exchange rate depreciation.",
    "Growth slowed and inflation eased towards the target band.",
]
CHECK = EXPORT + [
    "The Committee will continue to monitor developments.",
    "Credit to the private sector contracted in real terms, while non-performing loans remained "
    "elevated, and the Committee noted that risks to growth are tilted to the downside.",
]
enc = tokenizer(EXPORT, padding=True, return_tensors="pt")
wrapped = Logits(model).eval()      # .eval() matters: otherwise the export leaves dropout switched on
torch.onnx.export(
    wrapped,
    tuple(enc[n] for n in NAMES),
    FP32,
    dynamo=False,
    input_names=NAMES,
    output_names=["logits"],
    dynamic_axes={**{n: {0: "batch", 1: "sequence"} for n in NAMES}, "logits": {0: "batch"}},
    opset_version=17,
)
model.eval()

session = ort.InferenceSession(FP32)
np_enc = tokenizer(CHECK, padding=True, return_tensors="np")      # new batch size and length
onnx_logits = session.run(None, {n: np_enc[n].astype(np.int64) for n in NAMES})[0]
with torch.no_grad():
    torch_logits = model(**tokenizer(CHECK, padding=True, return_tensors="pt")).logits.numpy()
diff = np.abs(onnx_logits - torch_logits).max()
print("largest difference:", diff)
assert diff < 1e-3, "ONNX output differs from PyTorch - stop here"
print("CHECK 1 PASSED")

**Quantize** (make the model smaller) and check the smaller version gives the same labels as the full model on 500 sentences from the 2022-2026 split. It tries the smallest option first and keeps the first one that agrees on at least 97%.

In [ ]:
import pandas as pd
from huggingface_hub import snapshot_download

model.eval()
folder = snapshot_download(DATA_REPO, repo_type="dataset")
sample = pd.read_csv(f"{folder}/tone.csv").query("split == 'test'").sample(500, random_state=0)
texts = sample["text"].tolist()

with torch.no_grad():
    p_full = np.vstack([
        torch.softmax(model(**tokenizer(texts[i:i + 32], padding=True, truncation=True,
                                        max_length=128, return_tensors="pt")).logits, dim=-1).numpy()
        for i in range(0, len(texts), 32)
    ])
labels = [model.config.id2label[i] for i in range(3)]
h, d = labels.index("hawkish"), labels.index("dovish")
print("full-model probabilities for", len(texts), "sentences from 2022-2026")

In [ ]:
!pip -q install onnxconverter-common
import shutil
import onnx
from onnxconverter_common import float16
from onnxruntime.quantization import QuantType, quantize_dynamic
from onnxruntime.quantization.shape_inference import quant_pre_process

model.eval()
PRE = "export/model_pre.onnx"
quant_pre_process(FP32, PRE, skip_symbolic_shape=True)


def agreement(path):
    """Share of the 500 sentences with the same label as the full model, and mean net-tone change."""
    sess = ort.InferenceSession(path)
    def probs(batch):
        e = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="np")
        logits = sess.run(None, {n: e[n].astype(np.int64) for n in NAMES})[0].astype(np.float32)
        z = np.exp(logits - logits.max(axis=1, keepdims=True))
        return z / z.sum(axis=1, keepdims=True)
    p = np.vstack([probs(texts[i:i + 32]) for i in range(0, len(texts), 32)])
    agree = (p.argmax(1) == p_full.argmax(1)).mean()
    gap = np.abs((p[:, h] - p[:, d]) - (p_full[:, h] - p_full[:, d])).mean()
    return agree, gap


# Try the smaller options first; keep the first one that agrees on at least 97% of sentences.
options = [
    ("q8", "8-bit, per-channel", dict(per_channel=True, reduce_range=True, weight_type=QuantType.QInt8)),
    ("q8", "8-bit, per-channel, MatMul only",
     dict(per_channel=True, reduce_range=True, op_types_to_quantize=["MatMul"], weight_type=QuantType.QInt8)),
    ("fp16", "16-bit floats", None),
]
chosen = None
for i, (dtype, label, opts) in enumerate(options):
    path = f"export/candidate_{i}.onnx"
    if opts is None:
        onnx.save(float16.convert_float_to_float16(onnx.load(FP32), keep_io_types=True), path)
    else:
        quantize_dynamic(PRE, path, **opts)
    agree, gap = agreement(path)
    print(f"{label:34s} {os.path.getsize(path) / 1e6:5.0f} MB | same label {agree:.1%} | net-tone change {gap:.3f}")
    if agree >= 0.97:
        chosen = (dtype, path)
        break

assert chosen is not None, "no option agrees well enough - stop here and send me the lines above"
DTYPE, path = chosen
shutil.rmtree("export/onnx")
os.makedirs("export/onnx")
shutil.copy(path, "export/onnx/model_quantized.onnx" if DTYPE == "q8" else "export/onnx/model_fp16.onnx")
print("CHECK 2 PASSED - using", DTYPE, "->", os.listdir("export/onnx"))

**Model card.** Written honestly: the scores so far measure agreement with the rules, not accuracy.

In [ ]:
CARD = """---
language: en
base_model: ProsusAI/finbert
pipeline_tag: text-classification
library_name: transformers
tags:
- finance
- central-bank
- monetary-policy
- ghana
- transformers.js
---

# cb-policy-tone (v0)

Classifies a sentence from a Bank of Ghana Monetary Policy Committee statement as **hawkish**,
**neutral** or **dovish**. Part of the project
[github.com/Mbibachris/cb-policy-tone](https://github.com/Mbibachris/cb-policy-tone), with a demo at
[huggingface.co/spaces/Kobichris/policy-tone](https://huggingface.co/spaces/Kobichris/policy-tone).

## Training
- Base model: `ProsusAI/finbert`, fine-tuned for 3 classes (dovish, neutral, hawkish).
- Data: sentences from Bank of Ghana MPC statements, 2003-2026, split by date
  (train up to 2018, validation 2019-2021, test 2022-2026). Sentences announcing the rate decision
  are excluded.
- Labels: **weak labels** from a rule-based lexicon written by the author from a published codebook.
  The model therefore learns to imitate the rules.
- Class-weighted cross-entropy; 4 epochs; max length 128.

## Results so far
- Agreement with the rule labels (macro-F1): 0.74 on validation, 0.78 on the 2022-2026 test years.
  This is agreement with the rules, **not accuracy**.
- Accuracy against hand-labelled sentences: **pending** (a blind 400-sentence gold set is being labelled).
- Statement-level tone is associated with the next policy-rate decision in-sample, but does not improve
  out-of-sample forecasts beyond past decisions and inflation.

## Limitations
- Trained on one central bank's English statements; other banks or genres are untested.
- Inherits the lexicon's blind spots, e.g. rising inflation reads as hawkish even when inflation is low.
- Research prototype; not financial advice.

## Use
```python
from transformers import pipeline
clf = pipeline("text-classification", model="Kobichris/cb-policy-tone", top_k=None)
clf("Inflation pressures have intensified and risks are tilted to the upside.")
```
In the browser (transformers.js): `pipeline("text-classification", "Kobichris/cb-policy-tone", { dtype: "fp16" })`
uses `onnx/model_fp16.onnx` (16-bit, about 220 MB). 8-bit versions were tested and rejected:
they gave the same label as the full model on only about 84% of 500 test sentences.

Author: Christopher Mbiba.
"""
with open("export/README.md", "w") as f:
    f.write(CARD)
print("model card written")

In [ ]:
api.upload_folder(
    folder_path="export",
    repo_id=REPO,
    allow_patterns=["onnx/*.onnx", "README.md"],
    commit_message="Add quantized ONNX for transformers.js and a model card",
)
print(sorted(api.list_repo_files(REPO)))

**Make the model public.** The browser page can only download a public model.
What becomes public: the model weights, the tokenizer and the model card. **Not** the dataset (it stays
private) and not any sentence text. You can make it private again at any time
(model page -> Settings).

In [ ]:
api.update_repo_settings(REPO, private=False)
print("Public:", not api.model_info(REPO).private)